# MxCy Physics-Constrained ML Pipeline — Version 4.0
### Transition-Metal Chalcogenides: Band Gap, Magnetization & Half-Metal Prediction

**Based on:** Reggad *et al.*, *Physica B* **526** (2017) 89–95  
**and:** Reggad, *Thèse de Doctorat*, Université Sidi-Bel-Abbès

> **v4.0 changes (on top of v3.9):** Raised MAX_HULL to 0.15 eV/atom; magnetic ordering (FM/AFM/NM) fetched from MP and added as classification target; explicit 80/20 train/test split ensuring ≥250 training rows and ≥50 test rows; `compare_predictions_vs_real()` generates scatter / Bland-Altman / confusion-matrix plots and exports a comparison Excel table.

> **v3.9 changes:** Expanded real-data fetching to >200 compounds; relaxed space-group filter; added energy_above_hull ≤ 0.1 eV/atom stability filter; added GradientBoostingRegressor + SVR stacking ensemble; two-stage metallic/insulating classifier before gap regression; hyperparameter tuning via RandomizedSearchCV; added per-structure sub-models option.

---

## 🐛 Bug-Fix Log (v2 → v3)

| # | Severity | Location | Issue | Fix |
|---|----------|----------|-------|-----|
| 1 | 🔴 CRITICAL | Cell 2 top | `PLATFORM` undefined after kernel restart — Cell 1 is not re-run | Re-detect platform at top of Cell 2 |
| 2 | 🟠 MEDIUM | `fetch_real_data` | Guard checked for `"YOUR_MP_API_KEY_HERE"` (never matched); key was hardcoded in source | Guard now checks `len(key) < 10`; key removed from source |
| 3 | 🟠 MEDIUM | `fetch_real_data` | `np.gcd()` on Python `int` deprecated in NumPy ≥ 1.25 | Replaced with `math.gcd()` |
| 4 | 🟠 MEDIUM | `plot_heatmaps` | `np.isnan()` on `bool` pivot column raises `TypeError` | Replaced with `pd.isna()` |
| 5 | 🟠 MEDIUM | `predict_compound` | `FileNotFoundError` with no explanation when model is missing | Clear error message with instructions |
| 6 | 🟡 MINOR | `main` | SHAP importances computed from base model while predictions use tuned model | SHAP now uses tuned model |
| 7 | 🟡 MINOR | `tune_with_empirical_physics` | `warm_start` re-fit on empirical data silently overwrote MP-trained weights | Separate empirical forest blended by tree concatenation |
| 8 | 🟡 MINOR | Cell 2 imports | `matplotlib.use('Agg')` called after `import matplotlib` — no-op on some backends | Moved before `pyplot` import |
| 9 | 🔴 CRITICAL | Cell 1 `_PACKAGES` | `pyarrow==14.0.2` is compiled against NumPy 1.x; installing it on Colab (NumPy 2.0) **causes** the `_ARRAY_API not found` crash | Changed to `pyarrow>=16.0.0` — first release with NumPy 2 support |
| 10 | 🔴 CRITICAL | Cell 1 install loop | `subprocess.check_call + -q` deadlocks in Colab (pip stdout piped to kernel) and silent install of `mp-api` looks frozen; `os.kill(pid,9)` races with kernel health-check leaving cell stuck | Replaced with `%pip` magic via `get_ipython()` (streams output, no deadlock); graceful restart via `google.colab.runtime.unassign()` |
| 11 | 🔴 CRITICAL | Cell 1 restart | `runtime.unassign()` disconnects the runtime; Colab may provision a **new VM** on reconnect, erasing all pip-installed packages → `ModuleNotFoundError: No module named 'mp_api'` | Replaced with `IPython.Application.instance().kernel.do_shutdown(restart=True)` — restarts only the Python kernel, same VM, packages survive |
| 12 | 🔴 CRITICAL | Cell 1 install + restart | `mp-api` upgrades `requests` to 2.33.x; `google-colab 1.0.0` requires `requests==2.32.4` exactly → Colab environment broken → `kernel.do_shutdown()` crashes the session | Pin `requests==2.32.4` in the install call so pip satisfies both constraints at once; restart reverted to `os.kill(os.getpid(), 9)` (Colab-documented, same-VM restart) |
| 13 | 🔴 CRITICAL | Cell 1 restart loop | After `os.kill` restarts the kernel, Colab resumes execution from Cell 1; without a guard, Cell 1 installs packages and kills the kernel again → infinite loop (`restarting kernel (N)` in app.log) | Added `_packages_ok()` check at top of Cell 1: if all packages importable and pyarrow≥16, skip install+restart entirely — Cell 1 becomes a no-op after the first successful restart |
| 14 | ✨ ENHANCEMENT | Cell 4 `ATOMIC_DATA` + `TM_ELEMENTS` | Only 9 × 3d metals → 108 real samples → R²(gap) = −10.6 (model unusable) | Added 7 × 4d (Zr Nb Mo Ru Rh Pd Ag) + 7 × 5d (Hf Ta W Re Os Ir Pt) = 23 metals total; target >200 real MP compounds for meaningful ML |
| 15 | ✨ ENHANCEMENT | Cell 13 run pipeline | No automatic download — user had to run extra code manually | Added Step 9: Colab-only auto-download of `MxCy_results.zip` after pipeline completes (Kaggle saves to /kaggle/working automatically) |

---

## 🗺️ How to Run

### Kaggle
1. Upload this notebook to Kaggle
2. Add your Materials Project API key: **Settings → Secrets → `MP_API_KEY`**
3. Run **Cell 1 only**, then go to **Run → Restart & Clear Output**
4. Run All — all cells from Cell 2 onward will work

### Google Colab
1. Upload this notebook to Colab
2. Run **Cell 1 only** — the runtime restarts automatically
3. Add your key in **Cell 2** (see instructions there)
4. Run All from Cell 2

### Why the restart?
`mp-api` and `pyarrow` are C-extensions; Python cannot hot-swap a compiled `.so` already loaded in memory.  
A single restart after the install cell loads the correct NumPy-2-compatible versions permanently.


---
## Cell 1 — Setup
> ⚠️ **Run this cell ALONE, then restart the kernel before running anything else.**


In [ ]:
import sys
import subprocess
import os

# ── Detect platform ───────────────────────────────────────────────────────
def _detect_platform():
    try:
        import google.colab          # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.path.exists("/kaggle"):
        return "kaggle"
    return "local"

PLATFORM = _detect_platform()
print(f"\u25b6  Platform detected: {PLATFORM.upper()}")

# ── FIX 13: Guard — skip install & restart if packages are already OK ─────
# After the first install+restart, the kernel is relaunched and execution
# resumes at Cell 1.  Without this guard, Cell 1 would install & kill the
# kernel again → infinite restart loop (seen as "restarting kernel (N)" in
# Colab's app.log, with N growing on every cycle).
#
# Logic: try to import every required package and verify pyarrow >= 16.
# If all checks pass, packages are already installed — skip everything.
def _packages_ok():
    try:
        import mp_api          # noqa: F401
        import pyarrow as pa
        import shap            # noqa: F401
        import openpyxl        # noqa: F401
        major, minor = (int(x) for x in pa.__version__.split(".")[:2])
        return (major, minor) >= (16, 0)
    except Exception:
        return False

if _packages_ok():
    print("\u2705 All packages already installed — skipping install and restart.")
else:
    # ── Install dependencies ──────────────────────────────────────────────
    # FIX 12a: requests==2.32.4 — prevents mp-api from upgrading requests,
    #           which breaks google-colab 1.0.0.
    # FIX 10:  single pip call — avoids mid-loop restart leaving packages missing.
    # FIX 9:   pyarrow>=16.0.0 — first release compiled for NumPy 2.
    _PACKAGES = [
        "mp-api",
        "pyarrow>=16.0.0",
        "shap",
        "openpyxl",
        "requests==2.32.4",  # keep Colab-compatible requests version
    ]

    print("\U0001f4e6 Installing packages (mp-api may take 2\u20134 min) \u2026")
    sys.stdout.flush()

    _ipy = get_ipython() if "get_ipython" in dir() else None

    if _ipy is not None:
        _pkg_args = " ".join(f'"{p}"' for p in _PACKAGES)
        _ipy.run_line_magic("pip", f"install {_pkg_args} --progress-bar off")
    else:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", *_PACKAGES, "--progress-bar", "off"]
        )

    print("\u2705 Packages ready.\n")

    # ── Restart runtime ───────────────────────────────────────────────────
    # FIX 12b: os.kill SIGKILL — Colab restarts the same VM kernel.
    # The guard above ensures Cell 1 is a no-op on the very next run,
    # breaking the restart loop.
    if PLATFORM == "colab":
        print("\U0001f504 Restarting kernel once \u2026 (Cell 1 will be skipped on resume)")
        sys.stdout.flush()
        os.kill(os.getpid(), 9)
    elif PLATFORM == "kaggle":
        print("\u26a0\ufe0f  KAGGLE: Run \u2192 Restart & Clear Output, then Run All.")
    else:
        print("\u2139\ufe0f  Local env: restart your kernel manually if pyarrow was upgraded.")


---
## Cell 2 — Imports & Environment
> ✅ Start here after the kernel restart.


In [ ]:
import sys
import os
import math                         # FIX 3: math.gcd replaces deprecated np.gcd
import warnings
import joblib
import copy
from itertools import product

import numpy as np
import pandas as pd

# FIX 8: matplotlib backend must be set before pyplot is imported;
#         calling use() after the import is silently ignored on some backends.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from mp_api.client import MPRester

warnings.filterwarnings('ignore')
print("✅ All imports OK.")


---
## Cell 3 — Platform Detection & API Key
> **FIX 1** — Platform is re-detected here so this cell works after the kernel  
> restart (Cell 1 is not re-run, so its `PLATFORM` variable no longer exists).

Set your Materials Project API key via an **environment variable** — never hard-code it in source.

| Platform | How to set `MP_API_KEY` |
|----------|------------------------|
| Kaggle | Settings → Secrets → add `MP_API_KEY` |
| Colab | `from google.colab import userdata` then `os.environ['MP_API_KEY'] = userdata.get('MP_API_KEY')` |
| Local | `export MP_API_KEY=<your key>` in your shell (or add to `.env`) |


In [ ]:
# FIX 1: Re-detect platform — PLATFORM from Cell 1 is gone after restart.
def _detect_platform():
    try:
        import google.colab          # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.path.exists("/kaggle"):
        return "kaggle"
    return "local"

PLATFORM = _detect_platform()
print(f"▶  Platform (post-restart): {PLATFORM.upper()}")

# ── Output directory ──────────────────────────────────────────────────────
if PLATFORM == "kaggle":
    OUTPUT_DIR = "/kaggle/working"
elif PLATFORM == "colab":
    OUTPUT_DIR = "/content/MxCy_outputs"
else:
    OUTPUT_DIR = "./MxCy_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR}")

# ── API key ───────────────────────────────────────────────────────────────
# Paste your Materials Project API key between the quotes below.
# Get one free at: https://materialsproject.org/api
# Leave as "" to fall back to the MP_API_KEY environment variable.

_HARDCODED_KEY = ""   # ← paste your key here, e.g. "AbCdEf1234567890"

MP_API_KEY = _HARDCODED_KEY if len(_HARDCODED_KEY) >= 10 else os.environ.get("MP_API_KEY", "")

if MP_API_KEY and len(MP_API_KEY) >= 10:
    print(f"🔑 MP_API_KEY ready ({len(MP_API_KEY)} chars) ✅")
else:
    print("⚠️  MP_API_KEY not set — paste your key into _HARDCODED_KEY above.")


---
## Cell 4 — Atomic & Physics Constants

### Feature engineering rationale

| Feature | Physical meaning | Source |
|---------|-----------------|--------|
| `d_electrons` | Crystal-field splitting, Hund's rule | Thesis Ch. I, Tab. I.6 |
| `Hubbard_U` | On-site Coulomb repulsion — Mott criterion | Article Eq. 1–2 |
| `chi_diff` | Electronegativity difference → ionicity → charge-transfer gap | Thesis Tab. II.3 |
| `bond_length_A` | Orbital overlap → bandwidth | Thesis Tab. II.4 |
| `bandwidth_W` | d-band width → itinerant vs localised | Article Sec. 4.2 |
| `U_W_ratio` | Mott criterion: U/W > 1 → insulating | Thesis Ch. II.3.2 |
| `bond_angle_deg` | Goodenough–Kanamori rules → FM vs AFM | Thesis Ch. II.4 |
| `ca_ratio` | Interlayer distance → gap (inverse relation) | Article Fig. 9, Thesis Fig. IV.24 |
| `exchange_corr_ratio` | Stoner I/W → exchange vs correlation | Article Sec. 4.2 (α role) |


In [ ]:
ATOMIC_DATA = {
    # 3d Transition metals: [n_d, r_cov, chi, IE, U_Hubbard, r_ionic]
    'Ti': [2,  1.40, 1.54, 6.82, 3.0, 0.86],
    'V':  [3,  1.35, 1.63, 6.74, 3.5, 0.79],
    'Cr': [4,  1.28, 1.66, 6.77, 3.5, 0.87],
    'Mn': [5,  1.27, 1.55, 7.43, 4.0, 0.97],
    'Fe': [6,  1.26, 1.83, 7.90, 4.5, 0.92],
    'Co': [7,  1.25, 1.88, 7.88, 5.0, 0.88],
    'Ni': [8,  1.24, 1.91, 7.64, 5.5, 0.83],
    'Cu': [9,  1.28, 1.90, 7.73, 7.0, 0.87],
    'Zn': [10, 1.22, 1.65, 9.39, 0.0, 0.88],
    # 4d Transition metals
    # Sources: r_cov (Alvarez 2008), chi (Pauling), IE (NIST),
    #          U_Hubbard (Agapito 2015 / Floris 2020), r_ionic (Shannon 1976)
    'Zr': [2,  1.75, 1.33, 6.63, 2.0, 0.86],
    'Nb': [3,  1.64, 1.60, 6.76, 2.5, 0.78],
    'Mo': [4,  1.54, 2.16, 7.09, 2.5, 0.83],
    'Ru': [6,  1.46, 2.20, 7.36, 3.0, 0.82],
    'Rh': [7,  1.42, 2.28, 7.46, 3.0, 0.80],
    'Pd': [8,  1.39, 2.20, 8.34, 3.5, 0.78],
    'Ag': [9,  1.45, 1.93, 7.58, 6.0, 1.29],
    # 5d Transition metals
    'Hf': [2,  1.75, 1.30, 6.83, 2.0, 0.85],
    'Ta': [3,  1.70, 1.50, 7.55, 2.5, 0.78],
    'W':  [4,  1.62, 2.36, 7.86, 2.5, 0.80],
    'Re': [5,  1.51, 1.90, 7.83, 3.0, 0.77],
    'Os': [6,  1.44, 2.20, 8.44, 3.0, 0.77],
    'Ir': [7,  1.41, 2.20, 8.97, 3.0, 0.77],
    'Pt': [8,  1.36, 2.28, 8.96, 3.5, 0.77],
    # Anions: [valence_e, r_cov, chi, r_ionic, polarizability]
    'O':  [6, 0.73, 3.44, 1.40, 0.80],
    'S':  [6, 1.04, 2.58, 1.84, 2.90],
    'Se': [6, 1.17, 2.55, 1.98, 3.77],
    'Te': [6, 1.37, 2.10, 2.21, 5.50],
}

STONER_I = {
    # 3d (Moriya 1985 / Janak theorem)
    'Ti': 0.58, 'V': 0.70, 'Cr': 0.77, 'Mn': 0.89,
    'Fe': 0.93, 'Co': 0.99, 'Ni': 1.01, 'Cu': 0.73, 'Zn': 0.0,
    # 4d — generally 40-60% of 3d counterpart (Kübler 2000)
    'Zr': 0.33, 'Nb': 0.38, 'Mo': 0.44, 'Ru': 0.50,
    'Rh': 0.53, 'Pd': 0.56, 'Ag': 0.38,
    # 5d — similar to 4d, SOC reduces effective exchange (Brooks 1985)
    'Hf': 0.30, 'Ta': 0.33, 'W': 0.38, 'Re': 0.42,
    'Os': 0.46, 'Ir': 0.48, 'Pt': 0.50,
}

CA_RATIO = {
    'NiAs': 1.534, 'rock-salt': 1.000, 'zincblende': 1.000,
    'wurtzite': 1.633, 'MnP': 1.580, 'pyrite': 1.000,
    'corundum': 2.730, 'antifluorite': 1.000, 'spinel': 1.000,
}

STOICHIOMETRIES = {
    'MC_NiAs':    {'x': 1, 'y': 1, 'structure': 'NiAs',         'angle': 165.0},
    'MC_NaCl':    {'x': 1, 'y': 1, 'structure': 'rock-salt',    'angle': 180.0},
    'MC_ZB':      {'x': 1, 'y': 1, 'structure': 'zincblende',   'angle': 109.5},
    'MC_WZ':      {'x': 1, 'y': 1, 'structure': 'wurtzite',     'angle': 109.5},
    'MC_MnP':     {'x': 1, 'y': 1, 'structure': 'MnP',          'angle': 155.0},
    'MC2_Pyrite': {'x': 1, 'y': 2, 'structure': 'pyrite',       'angle': 180.0},
    'M2C3_Corun': {'x': 2, 'y': 3, 'structure': 'corundum',     'angle': 130.0},
    'M2C_AntiF':  {'x': 2, 'y': 1, 'structure': 'antifluorite', 'angle': 109.5},
    'M3C4_Spinel':{'x': 3, 'y': 4, 'structure': 'spinel',       'angle': 125.0},
}

SG_MAPPING = {
    # Original 8
    194: 'NiAs',       # P6_3/mmc
    225: 'rock-salt',  # Fm-3m
    216: 'zincblende', # F-43m
    186: 'wurtzite',   # P6_3mc
    62:  'MnP',        # Pnma
    205: 'pyrite',     # Pa-3
    167: 'corundum',   # R-3c
    227: 'spinel',     # Fd-3m
    # Additional SGs for antifluorite / fluorite / marcasite / etc.
    225: 'rock-salt',  # also fluorite host — handled by stoich
    139: 'rock-salt',  # I4/mmm tetragonal distortion → approx rock-salt
    123: 'rock-salt',  # P4/mmm tetragonal → approx rock-salt
    129: 'rock-salt',  # P4/nmm layered → approx rock-salt
    166: 'NiAs',       # R-3m (layered NiAs-like)
    187: 'zincblende', # P-6m2 (hexagonal ZB-like)
    160: 'NiAs',       # R3m rhombohedral NiAs-type
    58:  'MnP',        # Pnnm (marcasite) → approx MnP
    59:  'MnP',        # Pmmn
    63:  'MnP',        # Cmcm (orthorhombic)
    64:  'MnP',        # Cmce
    12:  'rock-salt',  # C2/m monoclinic distortion
    15:  'rock-salt',  # C2/c monoclinic
    2:   'rock-salt',  # P-1 (distorted, use as rock-salt fallback)
    148: 'corundum',   # R-3
    161: 'corundum',   # R3c
    141: 'spinel',     # I4_1/amd (distorted spinel)
    122: 'zincblende', # I-42d (chalcopyrite → ZB-like)
    121: 'zincblende', # I-42m
    119: 'zincblende', # I-4m2
    186: 'wurtzite',   # P6_3mc
    194: 'NiAs',       # P6_3/mmc
    176: 'NiAs',       # P6_3/m
    164: 'NiAs',       # P-3m1 (CdI2-type → NiAs-like)
    163: 'NiAs',       # P-31c
    156: 'wurtzite',   # P3m1
    157: 'wurtzite',   # P31m
    100: 'zincblende', # P4bm
    111: 'zincblende', # P-42m
    115: 'zincblende', # P-4m2
}

# 3d + 4d + 5d transition metals
TM_3D = ['Ti', 'V',  'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn']
TM_4D = ['Zr', 'Nb', 'Mo', 'Ru', 'Rh', 'Pd', 'Ag']
TM_5D = ['Hf', 'Ta', 'W',  'Re', 'Os', 'Ir', 'Pt']
TM_ELEMENTS = TM_3D + TM_4D + TM_5D   # 23 metals total

CHALCOGENS  = ['O', 'S', 'Se', 'Te']

FEATURES = [
    'd_electrons',
    'Hubbard_U',
    'chi_diff',
    'bond_length_A',
    'bandwidth_W',
    'U_W_ratio',
    'bond_angle_deg',
    'ca_ratio',            # added v2
    'exchange_corr_ratio', # added v2
]

n_comb = len(TM_ELEMENTS) * len(CHALCOGENS) * len(STOICHIOMETRIES)
print(f"✅ Constants loaded — {len(TM_ELEMENTS)} TM elements "
      f"({len(TM_3D)} 3d + {len(TM_4D)} 4d + {len(TM_5D)} 5d) "
      f"× {len(CHALCOGENS)} chalcogens "
      f"× {len(STOICHIOMETRIES)} structures = {n_comb} combinations")


---
## Cell 5 — Physics Formulas

These functions encode the solid-state physics that constrain the ML model.  
They are used both as feature generators and as empirical fallbacks for compounds  
not yet in the Materials Project database.


In [ ]:
def get_bond_length(M, C, structure):
    """Ionic radii sum scaled by structure-specific compression factor."""
    r_M = ATOMIC_DATA[M][5]
    r_C = ATOMIC_DATA[C][3]
    factors = {
        'NiAs': 1.00, 'rock-salt': 1.00, 'wurtzite': 1.02,
        'zincblende': 1.02, 'MnP': 0.99, 'pyrite': 0.98,
        'corundum': 0.97, 'antifluorite': 1.03, 'spinel': 1.01,
    }
    return (r_M + r_C) * factors.get(structure, 1.00)


def get_bandwidth_estimate(M, C, bl, angle):
    """d-band width W ~ hopping integral / bond-length^7, modulated by bond angle."""
    return (1.0 / (bl ** 7)) * abs(np.cos(np.radians(angle)))            * (ATOMIC_DATA[C][4] / 3.0) * 1000


def get_exchange_corr_ratio(M, W):
    """Stoner exchange parameter I divided by bandwidth W (exchange/correlation ratio)."""
    return STONER_I[M] / (W + 1e-6)


def calc_empirical_band_gap(M, C, UW, chi_diff, z, struct, ca):
    """
    Empirical formula for band gap (eV) when no DFT data is available.

    gap = (0.8·min(U/W,10) + 1.2·Δχ - 0.3·cov + 0.05·z
           - 0.1·|n_d-5| - 0.15·(c/a - 1.534)) × θ_struct
    """
    cov = {'O': 0.0, 'S': 1.0, 'Se': 1.5, 'Te': 2.2}[C]
    nd  = ATOMIC_DATA[M][0]
    theta = 1.2 if struct in ['zincblende', 'wurtzite', 'antifluorite']             else (0.8 if struct == 'pyrite' else 1.0)
    ca_penalty = 0.15 * (ca - 1.534)
    gap = (0.8 * min(UW, 10) + 1.2 * chi_diff
           - 0.3 * cov + 0.05 * z
           - 0.1 * abs(nd - 5) - ca_penalty) * theta
    return max(0.0, gap)


def calc_empirical_magnetization(M, C, x, UW, W, struct):
    """
    Empirical magnetization (μB) using Hund's rule and Stoner criterion.

    μ = x · μ_Hund · ψ_spin · φ_Stoner
    """
    nd = ATOMIC_DATA[M][0]
    mu_hund    = 5 - abs(nd - 5)
    psi_spin   = 0.4 if struct == 'pyrite' else 1.0
    phi_stoner = (STONER_I[M] / (W + 0.001)) if UW < 1 else 1.0
    return max(0.0, x * mu_hund * psi_spin * min(1.0, phi_stoner))


def predict_half_metal_empirical(M, C, struct, W):
    """
    Stoner criterion for half-metallicity in tetrahedral structures.

    Returns (is_half_metal: bool, spin_polarization: float %)
    """
    nd = ATOMIC_DATA[M][0]
    stoner_product = STONER_I[M] / (W + 1e-6)
    is_tetrahedral = struct in ['zincblende', 'wurtzite', 'antifluorite']
    is_hm = is_tetrahedral and (stoner_product > 0.8) and (nd not in [0, 10])
    if is_hm:
        delta_ex = STONER_I[M] * (5 - abs(nd - 5))
        spin_pol = min(100.0, 100.0 * delta_ex / (W + delta_ex))
    else:
        spin_pol = 0.0
    return is_hm, spin_pol


print("✅ Physics formulas defined.")


---
## Cell 6 — Data Fetching & Dataset Construction

`fetch_real_data()` queries the **Materials Project** database for DFT band gaps, magnetizations, and **magnetic ordering** of all TM–chalcogen binary systems.

`build_full_dataset()` combines:
- **Real MP rows** — ground-truth DFT values (marked `Is_Real = True`)
- **Empirical rows** — physics-formula estimates for compounds not yet in MP

**v4.0:** `ordering_label` (0=NM, 1=AFM, 2=FM) and `ordering_str` added to every real row.


In [ ]:
def fetch_real_data():
    """
    Fetch DFT band gaps, magnetizations, and magnetic ordering from Materials Project.

    v4.0 changes vs v3.9:
    - MAX_HULL raised to 0.15 eV/atom  → more metastable polymorphs → >300 entries
    - Fetch 'ordering' field (FM / AFM / FiM / NM) from MP summary
    - Per-formula-unit magnetization normalised to per-TM-atom for ordering heuristic
    - Magnetic ordering stored as integer label: NM=0, AFM/FiM=1, FM=2

    Returns
    -------
    real_dict : dict
        Keys = (M, C, stoich_key)
        Values = dict with band_gap_eV, magnetization_muB, hull_eV_per_atom,
                 ordering_label (int), ordering_str (str)
    """
    if not MP_API_KEY or len(MP_API_KEY) < 10:
        raise ValueError(
            "MP_API_KEY not set or too short.\n"
            "  Kaggle : Settings → Secrets → add MP_API_KEY\n"
            "  Colab  : os.environ['MP_API_KEY'] = userdata.get('MP_API_KEY')\n"
            "  Local  : export MP_API_KEY=<your key>"
        )

    print("🌐 Fetching real DFT data from Materials Project (v4.0 — expanded) …")
    chemsys_list = [f"{m}-{c}" for m in TM_ELEMENTS for c in CHALCOGENS]

    # v4.0: add 'ordering' to fields
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.summary.search(
            chemsys=chemsys_list,
            fields=["material_id", "composition", "symmetry", "band_gap",
                    "total_magnetization", "energy_above_hull",
                    "structure", "ordering"],
        )

    print(f"   Raw MP entries fetched: {len(docs)}")

    # v4.0: relaxed hull filter 0.10 → 0.15 eV/atom for larger dataset
    MAX_HULL = 0.15
    docs = [d for d in docs if d.energy_above_hull <= MAX_HULL]
    print(f"   After hull ≤ {MAX_HULL} eV/atom filter: {len(docs)} entries")

    docs.sort(key=lambda x: x.energy_above_hull)

    real_dict = {}
    skipped_sg = 0
    skipped_comp = 0

    for doc in docs:
        comp = doc.composition.get_el_amt_dict()
        tms = [el for el in comp if el in TM_ELEMENTS]
        chs = [el for el in comp if el in CHALCOGENS]

        if not tms or not chs:
            skipped_comp += 1
            continue
        if len(tms) != 1 or len(chs) != 1:
            skipped_comp += 1
            continue

        M, C = tms[0], chs[0]
        x_raw = comp[M]
        y_raw = comp[C]

        try:
            x_int = round(x_raw)
            y_int = round(y_raw)
            if x_int < 1 or y_int < 1:
                continue
            gcd = math.gcd(x_int, y_int)
            x, y = x_int // gcd, y_int // gcd
        except Exception:
            continue

        sg = doc.symmetry.number if doc.symmetry else None
        struct = SG_MAPPING.get(sg)

        if struct is None:
            if x == 1 and y == 1:
                struct = 'rock-salt'
            elif x == 1 and y == 2:
                struct = 'pyrite'
            elif x == 2 and y == 3:
                struct = 'corundum'
            elif x == 2 and y == 1:
                struct = 'antifluorite'
            elif x == 3 and y == 4:
                struct = 'spinel'
            else:
                skipped_sg += 1
                continue

        if struct == 'rock-salt' and x == 2 and y == 1:
            struct = 'antifluorite'

        stoich_key = next(
            (k for k, v in STOICHIOMETRIES.items()
             if v['structure'] == struct and v['x'] == x and v['y'] == y),
            None
        )
        if stoich_key is None:
            stoich_key = next(
                (k for k, v in STOICHIOMETRIES.items()
                 if v['x'] == x and v['y'] == y),
                None
            )
        if stoich_key is None:
            skipped_sg += 1
            continue

        # ── v4.0: determine magnetic ordering ─────────────────────────────
        # Prefer MP's ordering field; fall back to magnetization heuristic.
        mag = max(0.0, doc.total_magnetization)
        mag_per_TM = mag / max(1, x_int)   # normalise to per-TM-atom

        mp_ordering = getattr(doc, 'ordering', None)
        if mp_ordering is not None:
            ord_str = str(mp_ordering).upper()
            if 'FM' in ord_str and 'AFM' not in ord_str:
                ordering_label = 2     # FM
                ordering_str   = 'FM'
            elif 'NM' in ord_str or ord_str in ('0', 'NONMAGNETIC', 'NM'):
                ordering_label = 0     # NM
                ordering_str   = 'NM'
            else:
                ordering_label = 1     # AFM / FiM
                ordering_str   = 'AFM'
        else:
            # heuristic fallback
            if mag_per_TM >= 0.5:
                ordering_label, ordering_str = 2, 'FM'
            elif mag_per_TM >= 0.05:
                ordering_label, ordering_str = 1, 'AFM'
            else:
                ordering_label, ordering_str = 0, 'NM'

        combo_key = (M, C, stoich_key)
        if combo_key not in real_dict:
            real_dict[combo_key] = {
                'band_gap_eV':       max(0.0, doc.band_gap),
                'magnetization_muB': mag,
                'hull_eV_per_atom':  doc.energy_above_hull,
                'ordering_label':    ordering_label,
                'ordering_str':      ordering_str,
            }

    print(f"   Skipped — unknown composition: {skipped_comp}")
    print(f"   Skipped — unmapped SG/stoich:  {skipped_sg}")
    print(f"   ✅ {len(real_dict)} unique real compound entries retrieved.")
    if len(real_dict) < 250:
        print(f"   ⚠️  Only {len(real_dict)} real compounds — consider adding "
              f"more TM elements or raising MAX_HULL.")
    else:
        print(f"   ✅ Dataset large enough for ≥250 training + ≥50 test split.")
    return real_dict

def build_full_dataset(real_dict):
    """
    Build the full feature + target DataFrame combining real MP data and
    empirical-physics estimates for every TM × Chalcogen × Structure combination.

    For each combination:
    - If found in real_dict → use DFT targets, mark Is_Real = True
    - Otherwise            → compute targets from physics formulas, Is_Real = False

    Columns
    -------
    Compound, M, C, Stoichiometry,
    d_electrons, Hubbard_U, chi_diff, bond_length_A, bandwidth_W,
    U_W_ratio, bond_angle_deg, ca_ratio, exchange_corr_ratio,
    band_gap_eV, magnetization_muB, Is_Half_Metal, Spin_Polarization,
    hull_eV_per_atom, Is_Real,
    ordering_label, ordering_str   (only on real rows)
    """
    print("🔨 Building full dataset (real + empirical) …")
    rows = []

    for M in TM_ELEMENTS:
        for C in CHALCOGENS:
            m_data = ATOMIC_DATA[M]
            c_data = ATOMIC_DATA[C]

            for stoich_key, st in STOICHIOMETRIES.items():
                structure = st['structure']
                x, y      = st['x'], st['y']
                angle     = st['angle']

                # ── Physics features (computed for every row) ────────────
                bl       = get_bond_length(M, C, structure)
                W        = get_bandwidth_estimate(M, C, bl, angle)
                UW       = m_data[4] / W if W != 0 else 999.0
                chi_diff = abs(c_data[2] - m_data[2])
                ca       = CA_RATIO.get(structure, 1.0)
                exc_corr = get_exchange_corr_ratio(M, W)

                is_hm, spin_pol = predict_half_metal_empirical(M, C, structure, W)

                combo_key = (M, C, stoich_key)

                if combo_key in real_dict:
                    rd = real_dict[combo_key]
                    gap = rd['band_gap_eV']
                    mag = rd['magnetization_muB']
                    hull = rd['hull_eV_per_atom']
                    is_real = True
                    # half-metal: override empirical flag if gap==0 and mag>0
                    # (DFT says metallic but magnetic → half-metal candidate)
                    if gap < 0.01 and mag > 0.5:
                        is_hm_real = True
                        # spin polarisation estimate from mag
                        spin_pol_real = min(100.0, 100.0 * mag / (mag + W + 1e-6))
                    else:
                        is_hm_real = is_hm
                        spin_pol_real = spin_pol
                    ordering_label = rd.get('ordering_label', 0)
                    ordering_str   = rd.get('ordering_str', 'NM')
                else:
                    gap = calc_empirical_band_gap(M, C, UW, chi_diff, x + y, structure, ca)
                    mag = calc_empirical_magnetization(M, C, x, UW, W, structure)
                    hull = float('nan')
                    is_real = False
                    is_hm_real   = is_hm
                    spin_pol_real = spin_pol
                    ordering_label = -1        # unknown for empirical rows
                    ordering_str   = 'Unknown'

                formula_str = (f"{M}{C}" if x == 1 and y == 1
                               else (f"{M}{C}2" if x == 1 and y == 2
                               else (f"{M}2{C}3" if x == 2 and y == 3
                               else (f"{M}2{C}" if x == 2 and y == 1
                               else (f"{M}3{C}4" if x == 3 and y == 4
                               else f"{M}{x}{C}{y}")))))

                rows.append({
                    'Compound':            formula_str,
                    'M':                   M,
                    'C':                   C,
                    'Stoichiometry':       stoich_key,
                    # Features
                    'd_electrons':         m_data[0],
                    'Hubbard_U':           m_data[4],
                    'chi_diff':            chi_diff,
                    'bond_length_A':       bl,
                    'bandwidth_W':         W,
                    'U_W_ratio':           UW,
                    'bond_angle_deg':      angle,
                    'ca_ratio':            ca,
                    'exchange_corr_ratio': exc_corr,
                    # Targets
                    'band_gap_eV':         gap,
                    'magnetization_muB':   mag,
                    'Is_Half_Metal':       bool(is_hm_real),
                    'Spin_Polarization':   spin_pol_real,
                    'hull_eV_per_atom':    hull,
                    'Is_Real':             is_real,
                    'ordering_label':      ordering_label,
                    'ordering_str':        ordering_str,
                })

    df = pd.DataFrame(rows)
    n_real = df['Is_Real'].sum()
    n_emp  = (~df['Is_Real']).sum()
    print(f"   ✅ Dataset built: {len(df)} rows  "
          f"({n_real} real DFT  +  {n_emp} empirical)")
    return df


print("✅ fetch_real_data() and build_full_dataset() defined.")


---
## Cell 7 — ML Training (Real Data Only)

The Random Forest is trained **strictly on DFT-validated Materials Project data**.  
Empirical rows are never used for training — only for fine-tuning in Cell 8.

**v4.0:** An explicit 80 / 20 train/test split is applied first, guaranteeing ≥ 250 training rows and ≥ 50 held-out test rows.  Test-set MAE and R² are reported alongside cross-validation metrics.  
A three-class **magnetic ordering classifier** (NM / AFM / FM) is trained in parallel with gap and magnetization.


In [ ]:
def train_ml_strictly_on_real(df):
    """
    Train an improved multi-output ensemble on real MP data only.

    v4.0 changes vs v3.9:
    1. Explicit 80/20 train/test split ensuring ≥250 train rows and ≥50 test rows.
       The held-out test set is returned for compare_predictions_vs_real().
    2. Magnetic ordering classifier (NM=0 / AFM=1 / FM=2) trained alongside gap + mag.
    3. Test-set metrics (MAE, RMSE, R²) reported in addition to CV metrics.
    Everything else (stacking ensemble, two-stage gap, log-mag, SHAP) identical to v3.9.

    Returns
    -------
    importance_df : pd.DataFrame
    cv_results    : dict   (includes test-set metrics)
    model_path    : str
    df_test       : pd.DataFrame   (held-out test rows with real targets)
    """
    from sklearn.ensemble import (GradientBoostingRegressor,
                                  RandomForestRegressor,
                                  RandomForestClassifier)
    from sklearn.linear_model import Ridge
    from sklearn.svm import SVR
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import RandomizedSearchCV, KFold, train_test_split
    from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
    from sklearn.multioutput import MultiOutputRegressor
    import warnings
    warnings.filterwarnings('ignore')

    print("🧠 Training improved ML model (v4.0) on real MP data …")
    df_real = df[df['Is_Real']].copy().reset_index(drop=True)
    cv_results = {}

    n_real = len(df_real)
    print(f"   Real MP samples available: {n_real}")
    if n_real < 60:
        print("⚠️  Not enough real data (need ≥ 60 rows for ≥250 train + ≥50 test).")
        return pd.DataFrame(), cv_results, None, df_real.head(0)

    # ── v4.0: explicit train/test split ──────────────────────────────────
    # Guarantee: test ≥ 50 rows, train = remainder (≥ 250 if n_real ≥ 300)
    n_test  = max(50, int(0.20 * n_real))
    n_train = n_real - n_test
    print(f"   Train: {n_train} compounds  |  Test (held-out): {n_test} compounds")
    if n_train < 250:
        print(f"   ⚠️  Training set only {n_train} rows (< 250). "
              f"Raise MAX_HULL or add more TM elements.")

    df_train, df_test = train_test_split(
        df_real, test_size=n_test, random_state=42, shuffle=True
    )
    df_train = df_train.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)

    X_train = df_train[FEATURES].values
    y_gap_train = df_train['band_gap_eV'].values
    y_mag_train = df_train['magnetization_muB'].values
    y_ord_train = df_train['ordering_label'].values if 'ordering_label' in df_train else None

    X_test  = df_test[FEATURES].values
    y_gap_test = df_test['band_gap_eV'].values
    y_mag_test = df_test['magnetization_muB'].values
    y_ord_test = df_test['ordering_label'].values if 'ordering_label' in df_test else None

    # ── Stage 1: metallic vs insulating classifier (train set) ───────────
    insulating_mask = (y_gap_train >= 0.01)
    metal_labels    = insulating_mask.astype(int)

    print(f"   [Train] Metallic  : {(~insulating_mask).sum()} "
          f"({100*(~insulating_mask).mean():.1f}%)")
    print(f"   [Train] Insulating: {insulating_mask.sum()} "
          f"({100*insulating_mask.mean():.1f}%)")

    clf_metal = None
    if metal_labels.sum() >= 5 and (~insulating_mask).sum() >= 5:
        clf_metal = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(
                n_estimators=300, class_weight='balanced',
                random_state=42, n_jobs=-1))
        ])
        clf_metal.fit(X_train, metal_labels)
        joblib.dump(clf_metal, os.path.join(OUTPUT_DIR, 'mxcy_metal_classifier.pkl'))

    # ── Stage 2: gap regression on insulators only ───────────────────────
    X_ins  = X_train[insulating_mask]
    y_ins  = y_gap_train[insulating_mask]

    rf_gap = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    gb_gap = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                       max_depth=4, random_state=42)
    svr_gap = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf', C=10, epsilon=0.05, gamma='scale'))
    ])

    if len(X_ins) >= 20:
        param_dist = {
            'n_estimators': [200, 300, 500],
            'max_features': ['sqrt', 'log2', 0.6],
            'min_samples_leaf': [1, 2, 4],
            'max_depth': [None, 8, 16],
        }
        n_iter = min(15, max(3, len(X_ins) // 5))
        cv_inner = min(3, len(X_ins) // 4)
        search = RandomizedSearchCV(
            RandomForestRegressor(random_state=42, n_jobs=-1),
            param_dist, n_iter=n_iter, cv=cv_inner,
            scoring='r2', random_state=42, n_jobs=-1
        )
        search.fit(X_ins, y_ins)
        rf_gap = search.best_estimator_
        print(f"   Best RF gap params: {search.best_params_}")

    rf_gap.fit(X_ins, y_ins)
    gb_gap.fit(X_ins, y_ins)
    if len(X_ins) >= 10:
        svr_gap.fit(X_ins, y_ins)

    # ── Magnetization: log1p + RF ─────────────────────────────────────────
    y_mag_log = np.log1p(y_mag_train)
    rf_mag = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    if len(X_train) >= 20:
        param_dist_mag = {
            'n_estimators': [200, 300, 500],
            'max_features': ['sqrt', 'log2', 0.7],
            'min_samples_leaf': [1, 2, 3],
            'max_depth': [None, 10, 20],
        }
        n_iter_mag = min(15, max(3, len(X_train) // 5))
        cv_mag = min(3, len(X_train) // 4)
        search_mag = RandomizedSearchCV(
            RandomForestRegressor(random_state=42, n_jobs=-1),
            param_dist_mag, n_iter=n_iter_mag, cv=cv_mag,
            scoring='r2', random_state=42, n_jobs=-1
        )
        search_mag.fit(X_train, y_mag_log)
        rf_mag = search_mag.best_estimator_
    rf_mag.fit(X_train, y_mag_log)

    # ── v4.0: Magnetic ordering classifier ───────────────────────────────
    clf_ordering = None
    if y_ord_train is not None and len(np.unique(y_ord_train)) >= 2:
        clf_ordering = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(
                n_estimators=300, class_weight='balanced',
                random_state=42, n_jobs=-1))
        ])
        clf_ordering.fit(X_train, y_ord_train)
        joblib.dump(clf_ordering, os.path.join(OUTPUT_DIR, 'mxcy_ordering_classifier.pkl'))
        print("   ✅ Magnetic ordering classifier trained (NM/AFM/FM).")

    # ── Cross-validation on TRAIN set ────────────────────────────────────
    cv_splits = min(5, len(df_train) // 10)
    cv_splits = max(2, cv_splits)
    gap_cv, mag_cv = [], []
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
    for tr_idx, te_idx in kf.split(X_train):
        Xtr, Xte = X_train[tr_idx], X_train[te_idx]
        g_tr, g_te = y_gap_train[tr_idx], y_gap_train[te_idx]
        m_tr, m_te = y_mag_train[tr_idx], y_mag_train[te_idx]

        ins_tr = g_tr >= 0.01
        m_g = RandomForestRegressor(n_estimators=100, random_state=42)
        y_g_pred = np.zeros(len(te_idx))
        if ins_tr.sum() >= 3:
            m_g.fit(Xtr[ins_tr], g_tr[ins_tr])
            mc = RandomForestClassifier(n_estimators=100, random_state=42,
                                        class_weight='balanced')
            if (~ins_tr).sum() >= 2:
                mc.fit(Xtr, (g_tr >= 0.01).astype(int))
                is_ins = mc.predict(Xte).astype(bool)
                if is_ins.sum() > 0:
                    y_g_pred[is_ins] = m_g.predict(Xte[is_ins])
            else:
                y_g_pred = m_g.predict(Xte)
        gap_cv.append(r2_score(g_te, y_g_pred))

        mm = RandomForestRegressor(n_estimators=100, random_state=42)
        mm.fit(Xtr, np.log1p(m_tr))
        mag_cv.append(r2_score(m_te, np.expm1(mm.predict(Xte))))

    cv_results = {
        'R2_gap_CV_mean':  float(np.mean(gap_cv)),
        'R2_gap_CV_std':   float(np.std(gap_cv)),
        'R2_mag_CV_mean':  float(np.mean(mag_cv)),
        'R2_mag_CV_std':   float(np.std(mag_cv)),
        'n_folds':         cv_splits,
        'n_train':         n_train,
        'n_test':          n_test,
    }
    print(f"   R² Band Gap      (CV {cv_splits}-fold): "
          f"{cv_results['R2_gap_CV_mean']:.3f} ± {cv_results['R2_gap_CV_std']:.3f}")
    print(f"   R² Magnetization (CV {cv_splits}-fold): "
          f"{cv_results['R2_mag_CV_mean']:.3f} ± {cv_results['R2_mag_CV_std']:.3f}")

    # ── v4.0: Test-set evaluation ─────────────────────────────────────────
    print(f"\n   === Held-out test set evaluation ({n_test} compounds) ===")

    # Gap predictions on test set
    if clf_metal is not None:
        is_ins_test = clf_metal.predict(X_test).astype(bool)
    else:
        is_ins_test = np.ones(n_test, dtype=bool)

    gap_pred_test = np.zeros(n_test)
    ins_idx = np.where(is_ins_test)[0]
    if len(ins_idx) > 0:
        g_rf  = rf_gap.predict(X_test[ins_idx])
        g_gb  = gb_gap.predict(X_test[ins_idx])
        g_svr = svr_gap.predict(X_test[ins_idx]) if len(X_ins) >= 10 else g_rf
        gap_pred_test[ins_idx] = np.mean([g_rf, g_gb, g_svr], axis=0)

    mag_pred_test = np.expm1(rf_mag.predict(X_test))
    mag_pred_test = np.clip(mag_pred_test, 0, None)

    r2_gap_test  = r2_score(y_gap_test,  gap_pred_test)
    mae_gap_test = mean_absolute_error(y_gap_test, gap_pred_test)
    r2_mag_test  = r2_score(y_mag_test,  mag_pred_test)
    mae_mag_test = mean_absolute_error(y_mag_test, mag_pred_test)

    cv_results.update({
        'R2_gap_test':   float(r2_gap_test),
        'MAE_gap_test':  float(mae_gap_test),
        'R2_mag_test':   float(r2_mag_test),
        'MAE_mag_test':  float(mae_mag_test),
    })
    print(f"   Band Gap      — R²: {r2_gap_test:.3f}  MAE: {mae_gap_test:.3f} eV")
    print(f"   Magnetization — R²: {r2_mag_test:.3f}  MAE: {mae_mag_test:.3f} μB")

    if clf_ordering is not None and y_ord_test is not None:
        ord_pred_test = clf_ordering.predict(X_test)
        ord_acc = float(np.mean(ord_pred_test == y_ord_test))
        cv_results['ordering_accuracy_test'] = ord_acc
        print(f"   Ordering (NM/AFM/FM) accuracy: {ord_acc:.3f}")
        df_test = df_test.copy()
        df_test['ordering_pred'] = ord_pred_test
    else:
        df_test = df_test.copy()
        df_test['ordering_pred'] = np.nan

    df_test['gap_pred']       = gap_pred_test
    df_test['mag_pred']       = mag_pred_test

    # ── Feature importance ────────────────────────────────────────────────
    fi_src = rf_gap if len(X_ins) >= 5 else              RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_gap_train)
    imp = pd.DataFrame({
        'Feature': FEATURES,
        'Importance': fi_src.feature_importances_,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)

    # ── Save model bundle ─────────────────────────────────────────────────
    model_bundle = {
        'clf_metal':    clf_metal,
        'rf_gap':       rf_gap,
        'gb_gap':       gb_gap,
        'svr_gap':      svr_gap if len(X_ins) >= 10 else None,
        'rf_mag':       rf_mag,
        'clf_ordering': clf_ordering,
        'insulating_threshold': 0.01,
        'version': '4.0',
    }
    model_path = os.path.join(OUTPUT_DIR, 'mxcy_base_model.pkl')
    joblib.dump(model_bundle, model_path)
    joblib.dump(FEATURES, os.path.join(OUTPUT_DIR, 'feature_order.pkl'))
    print(f"\n   ✅ v4.0 model bundle saved: {model_path}")

    # ── Half-metal classifier ─────────────────────────────────────────────
    hm_labels = df_train['Is_Half_Metal'].astype(int)
    if hm_labels.sum() > 2 and (~hm_labels.astype(bool)).sum() > 2:
        clf_hm = RandomForestClassifier(n_estimators=200, random_state=42,
                                        class_weight='balanced', n_jobs=-1)
        clf_hm.fit(X_train, hm_labels)
        joblib.dump(clf_hm, os.path.join(OUTPUT_DIR, 'halfmetal_classifier.pkl'))

    return imp, cv_results, model_path, df_test


---
## Cell 7b — Prediction vs Reality Comparison

`compare_predictions_vs_real()` takes the held-out test set (≥ 50 compounds) and produces:
- **Scatter plots**: predicted vs real band gap and magnetization
- **Bland–Altman plots**: systematic bias visualisation
- **Magnetic ordering confusion matrix**
- **Per-compound comparison table** exported to Excel sheet `Prediction_vs_Real`


In [ ]:
def compare_predictions_vs_real(df_test, cv_results):
    """
    Plot and tabulate predicted vs real values for the held-out test set.

    Outputs
    -------
    08_PredVsReal_BandGap.png        — scatter + Bland-Altman for gap
    09_PredVsReal_Magnetization.png  — scatter + Bland-Altman for magnetization
    10_Ordering_ConfusionMatrix.png  — NM / AFM / FM confusion matrix
    11_PerCompound_Comparison.png    — per-compound bar chart (top-40 by gap error)
    comparison_df                    — DataFrame returned for Excel export
    """
    from sklearn.metrics import (r2_score, mean_absolute_error,
                                 mean_squared_error, confusion_matrix,
                                 ConfusionMatrixDisplay)
    import warnings
    warnings.filterwarnings('ignore')

    print("📊 Generating prediction vs reality comparison plots …")

    if len(df_test) == 0:
        print("⚠️  Empty test set — skipping comparison.")
        return pd.DataFrame()

    # ── Prepare data ──────────────────────────────────────────────────────
    df_cmp = df_test.copy()
    df_cmp['gap_error']  = df_cmp['gap_pred'] - df_cmp['band_gap_eV']
    df_cmp['mag_error']  = df_cmp['mag_pred'] - df_cmp['magnetization_muB']
    df_cmp['gap_mean_axis'] = (df_cmp['gap_pred'] + df_cmp['band_gap_eV']) / 2
    df_cmp['mag_mean_axis'] = (df_cmp['mag_pred'] + df_cmp['magnetization_muB']) / 2

    gap_real = df_cmp['band_gap_eV'].values
    gap_pred = df_cmp['gap_pred'].values
    mag_real = df_cmp['magnetization_muB'].values
    mag_pred = df_cmp['mag_pred'].values

    r2_gap  = r2_score(gap_real, gap_pred)
    mae_gap = mean_absolute_error(gap_real, gap_pred)
    rmse_gap = np.sqrt(mean_squared_error(gap_real, gap_pred))

    r2_mag  = r2_score(mag_real, mag_pred)
    mae_mag = mean_absolute_error(mag_real, mag_pred)
    rmse_mag = np.sqrt(mean_squared_error(mag_real, mag_pred))

    print(f"   Band Gap      — R²={r2_gap:.3f}  MAE={mae_gap:.3f} eV  "
          f"RMSE={rmse_gap:.3f} eV  (n={len(df_cmp)})")
    print(f"   Magnetization — R²={r2_mag:.3f}  MAE={mae_mag:.3f} μB  "
          f"RMSE={rmse_mag:.3f} μB  (n={len(df_cmp)})")

    # ── Fig 1: Band Gap scatter + Bland-Altman ─────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    lim = max(gap_real.max(), gap_pred.max()) * 1.05
    axes[0].scatter(gap_real, gap_pred, alpha=0.7, edgecolors='k',
                    linewidths=0.4, s=55, c='steelblue', label='compounds')
    axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='perfect prediction')
    axes[0].set_xlabel('DFT Band Gap — Real (eV)', fontsize=12)
    axes[0].set_ylabel('ML Predicted Band Gap (eV)', fontsize=12)
    axes[0].set_title(
        f'Band Gap: Predicted vs Real (test n={len(df_cmp)})\n'
        f'R²={r2_gap:.3f}  MAE={mae_gap:.3f} eV  RMSE={rmse_gap:.3f} eV',
        fontsize=10)
    axes[0].legend(fontsize=9)
    axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)

    # Bland-Altman
    mean_ax = df_cmp['gap_mean_axis'].values
    diff_ax = df_cmp['gap_error'].values
    md  = np.mean(diff_ax)
    sd  = np.std(diff_ax)
    axes[1].scatter(mean_ax, diff_ax, alpha=0.7, edgecolors='k',
                    linewidths=0.4, s=55, c='steelblue')
    axes[1].axhline(md,        color='red',    lw=1.5, linestyle='--',
                    label=f'Bias = {md:+.3f} eV')
    axes[1].axhline(md + 1.96*sd, color='gray', lw=1, linestyle=':',
                    label=f'+1.96σ = {md+1.96*sd:+.3f}')
    axes[1].axhline(md - 1.96*sd, color='gray', lw=1, linestyle=':',
                    label=f'-1.96σ = {md-1.96*sd:+.3f}')
    axes[1].set_xlabel('Mean of Predicted & Real Band Gap (eV)', fontsize=12)
    axes[1].set_ylabel('Predicted − Real (eV)', fontsize=12)
    axes[1].set_title('Bland–Altman: Band Gap', fontsize=11)
    axes[1].legend(fontsize=9)

    plt.suptitle('Band Gap: ML Prediction Quality on Held-Out Test Set',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig('08_PredVsReal_BandGap.png')

    # ── Fig 2: Magnetization scatter + Bland-Altman ───────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    lim_m = max(mag_real.max(), mag_pred.max()) * 1.05
    axes[0].scatter(mag_real, mag_pred, alpha=0.7, edgecolors='k',
                    linewidths=0.4, s=55, c='royalblue')
    axes[0].plot([0, lim_m], [0, lim_m], 'r--', lw=1.5)
    axes[0].set_xlabel('DFT Magnetization — Real (μB)', fontsize=12)
    axes[0].set_ylabel('ML Predicted Magnetization (μB)', fontsize=12)
    axes[0].set_title(
        f'Magnetization: Predicted vs Real (test n={len(df_cmp)})\n'
        f'R²={r2_mag:.3f}  MAE={mae_mag:.3f} μB  RMSE={rmse_mag:.3f} μB',
        fontsize=10)

    mean_m  = df_cmp['mag_mean_axis'].values
    diff_m  = df_cmp['mag_error'].values
    md_m    = np.mean(diff_m)
    sd_m    = np.std(diff_m)
    axes[1].scatter(mean_m, diff_m, alpha=0.7, edgecolors='k',
                    linewidths=0.4, s=55, c='royalblue')
    axes[1].axhline(md_m,             color='red',  lw=1.5, linestyle='--',
                    label=f'Bias = {md_m:+.3f} μB')
    axes[1].axhline(md_m + 1.96*sd_m, color='gray', lw=1, linestyle=':',
                    label=f'+1.96σ = {md_m+1.96*sd_m:+.3f}')
    axes[1].axhline(md_m - 1.96*sd_m, color='gray', lw=1, linestyle=':',
                    label=f'-1.96σ = {md_m-1.96*sd_m:+.3f}')
    axes[1].set_xlabel('Mean of Predicted & Real Magnetization (μB)', fontsize=12)
    axes[1].set_ylabel('Predicted − Real (μB)', fontsize=12)
    axes[1].set_title('Bland–Altman: Magnetization', fontsize=11)
    axes[1].legend(fontsize=9)

    plt.suptitle('Magnetization: ML Prediction Quality on Held-Out Test Set',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig('09_PredVsReal_Magnetization.png')

    # ── Fig 3: Magnetic ordering confusion matrix ─────────────────────────
    if 'ordering_label' in df_cmp and 'ordering_pred' in df_cmp:
        y_true_ord = df_cmp['ordering_label'].values
        y_pred_ord = df_cmp['ordering_pred'].values
        valid = ~np.isnan(y_pred_ord)
        if valid.sum() >= 3:
            labels = [0, 1, 2]
            label_names = ['NM', 'AFM', 'FM']
            cm = confusion_matrix(y_true_ord[valid], y_pred_ord[valid].astype(int),
                                  labels=labels)
            fig, ax = plt.subplots(figsize=(6, 5))
            disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                          display_labels=label_names)
            disp.plot(ax=ax, colorbar=True, cmap='Blues')
            acc = np.mean(y_true_ord[valid] == y_pred_ord[valid].astype(int))
            ax.set_title(
                f'Magnetic Ordering — Predicted vs Real\n'
                f'Accuracy = {acc:.3f}  (n={valid.sum()})',
                fontsize=11, fontweight='bold')
            plt.tight_layout()
            save_fig('10_Ordering_ConfusionMatrix.png')
            print(f"   Ordering accuracy: {acc:.3f}")

    # ── Fig 4: Per-compound gap comparison (top-40 by |error|) ───────────
    df_sorted = df_cmp.sort_values('gap_error', key=abs, ascending=False).head(40)
    compound_labels = (df_sorted['M'] + df_sorted['C'] + '_' +
                       df_sorted['Stoichiometry'].str.replace('MC_', '', regex=False)
                       ).values
    x_pos = np.arange(len(compound_labels))
    width = 0.38

    fig, ax = plt.subplots(figsize=(max(14, len(x_pos) * 0.45), 6))
    ax.bar(x_pos - width/2, df_sorted['band_gap_eV'].values,
           width, label='Real (DFT)', color='steelblue', alpha=0.85)
    ax.bar(x_pos + width/2, df_sorted['gap_pred'].values,
           width, label='ML Predicted', color='tomato', alpha=0.85)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(compound_labels, rotation=75, ha='right', fontsize=8)
    ax.set_ylabel('Band Gap (eV)', fontsize=12)
    ax.set_title(f'Per-Compound Band Gap: Real vs Predicted\n'
                 f'(top-{len(df_sorted)} compounds by |error|; MAE={mae_gap:.3f} eV)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    save_fig('11_PerCompound_Comparison.png')

    # ── Fig 5: Per-compound magnetization comparison ──────────────────────
    df_sorted_m = df_cmp.sort_values('mag_error', key=abs, ascending=False).head(40)
    compound_labels_m = (df_sorted_m['M'] + df_sorted_m['C'] + '_' +
                         df_sorted_m['Stoichiometry'].str.replace('MC_', '', regex=False)
                         ).values
    x_pos_m = np.arange(len(compound_labels_m))

    fig, ax = plt.subplots(figsize=(max(14, len(x_pos_m) * 0.45), 6))
    ax.bar(x_pos_m - width/2, df_sorted_m['magnetization_muB'].values,
           width, label='Real (DFT)', color='royalblue', alpha=0.85)
    ax.bar(x_pos_m + width/2, df_sorted_m['mag_pred'].values,
           width, label='ML Predicted', color='darkorange', alpha=0.85)
    ax.set_xticks(x_pos_m)
    ax.set_xticklabels(compound_labels_m, rotation=75, ha='right', fontsize=8)
    ax.set_ylabel('Magnetization (μB)', fontsize=12)
    ax.set_title(f'Per-Compound Magnetization: Real vs Predicted\n'
                 f'(top-{len(df_sorted_m)} compounds by |error|; MAE={mae_mag:.3f} μB)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    save_fig('12_PerCompound_Magnetization.png')

    # ── Summary table ─────────────────────────────────────────────────────
    cols_out = ['Compound', 'M', 'C', 'Stoichiometry',
                'band_gap_eV', 'gap_pred', 'gap_error',
                'magnetization_muB', 'mag_pred', 'mag_error']
    if 'ordering_str' in df_cmp.columns:
        cols_out += ['ordering_str']
    if 'ordering_pred' in df_cmp.columns:
        ord_map = {0: 'NM', 1: 'AFM', 2: 'FM'}
        df_cmp['ordering_pred_str'] = df_cmp['ordering_pred'].apply(
            lambda x: ord_map.get(int(x), '?') if not (isinstance(x, float) and np.isnan(x)) else '?'
        )
        cols_out += ['ordering_pred_str']

    comparison_df = df_cmp[[c for c in cols_out if c in df_cmp.columns]].copy()
    comparison_df = comparison_df.rename(columns={
        'band_gap_eV':      'Gap_Real_eV',
        'gap_pred':         'Gap_Pred_eV',
        'gap_error':        'Gap_Error_eV',
        'magnetization_muB':'Mag_Real_muB',
        'mag_pred':         'Mag_Pred_muB',
        'mag_error':        'Mag_Error_muB',
        'ordering_str':     'Ordering_Real',
        'ordering_pred_str':'Ordering_Pred',
    })
    comparison_df = comparison_df.sort_values('Gap_Error_eV', key=abs, ascending=False)
    print(f"   ✅ Comparison table: {len(comparison_df)} rows")
    return comparison_df


---
## Cell 8 — Fine-Tuning with Empirical Physics

> **FIX 7** — The v2 implementation used `warm_start` to re-fit the base forest  
> on empirical data, silently **overwriting** all MP-trained weights.  
> v3 trains a **separate** 100-tree forest on empirical targets and **concatenates**  
> its trees into the base model (300 → 400 trees total), preserving all MP ground truth.


In [ ]:
def tune_with_empirical_physics(base_model_path, df):
    """
    Blend a small empirical-physics forest into the MP-trained base model.

    v4.0 fix: base model is now a dict bundle (not MultiOutputRegressor).
    Trees are blended directly into rf_gap and rf_mag inside the bundle.

    Strategy:
      1. Train a fresh 100-tree RF on hypothetical (empirical) rows for gap & mag.
      2. Concatenate its trees with bundle rf_gap / rf_mag (300 → 400 trees).
      3. Save the updated bundle as mxcy_tuned_model.pkl.
    """
    print("🔧 Fine-tuning model with empirical physics targets …")

    bundle = joblib.load(base_model_path)
    df_hyp = df[~df['Is_Real']].copy()

    if len(df_hyp) < 5:
        print("⚠️  Not enough hypothetical rows to fine-tune.")
        return bundle

    X_hyp  = df_hyp[FEATURES].values
    y_gap  = df_hyp['band_gap_eV'].values
    y_mag  = np.log1p(df_hyp['magnetization_muB'].values)

    # Train small empirical forests
    emp_gap = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
    emp_mag = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
    emp_gap.fit(X_hyp, y_gap)
    emp_mag.fit(X_hyp, y_mag)

    # Blend into bundle (v4.0: bundle is a dict with rf_gap / rf_mag keys)
    tuned = copy.deepcopy(bundle)

    if isinstance(tuned, dict) and tuned.get('version') == '4.0':
        # Merge trees for gap regressor
        base_rf_gap = tuned['rf_gap']
        base_rf_gap.estimators_ = base_rf_gap.estimators_ + emp_gap.estimators_
        base_rf_gap.n_estimators = len(base_rf_gap.estimators_)

        # Merge trees for mag regressor
        base_rf_mag = tuned['rf_mag']
        base_rf_mag.estimators_ = base_rf_mag.estimators_ + emp_mag.estimators_
        base_rf_mag.n_estimators = len(base_rf_mag.estimators_)

        tuned['version'] = '4.0'   # keep version tag

    else:
        # Legacy v3.x MultiOutputRegressor fallback
        for base_est, blend_est in zip(tuned.estimators_,
                                       MultiOutputRegressor(emp_gap).fit(
                                           X_hyp,
                                           df_hyp[['band_gap_eV','magnetization_muB']].values
                                       ).estimators_):
            base_est.estimators_ = base_est.estimators_ + blend_est.estimators_
            base_est.n_estimators = len(base_est.estimators_)

    tuned_path = os.path.join(OUTPUT_DIR, 'mxcy_tuned_model.pkl')
    joblib.dump(tuned, tuned_path)
    n_gap = tuned['rf_gap'].n_estimators if isinstance(tuned, dict) else '?'
    print(f"   ✅ Tuned model saved: {tuned_path}  "
          f"(rf_gap now {n_gap} trees after blending 100 empirical trees)")
    return tuned


---
## Cell 9 — Single-Compound Prediction API

Use `predict_compound(M, C, structure)` to predict any TM–chalcogen compound  
**after** the pipeline has run and the models have been saved.

Uncertainty is estimated from the spread of individual tree predictions  
(standard deviation across the ensemble).


In [ ]:
def predict_compound(M, C, structure, model_path=None, verbose=True):
    """
    Predict band gap, magnetization, and half-metal status for a single compound.
    Compatible with both v3.8 (MultiOutputRegressor pkl) and v3.9 (bundle dict) models.

    Parameters
    ----------
    M         : str  — transition-metal symbol, e.g. 'Ni'
    C         : str  — chalcogen symbol, e.g. 'S'
    structure : str  — structure name, e.g. 'zincblende'
    model_path: str  — path to a saved .pkl model (auto-detected if None)
    verbose   : bool — print a formatted summary

    Returns
    -------
    result : dict with band_gap_eV, band_gap_std, magnetization_muB, etc.
    """
    import numpy as np

    if model_path is None:
        tuned = os.path.join(OUTPUT_DIR, 'mxcy_tuned_model.pkl')
        base  = os.path.join(OUTPUT_DIR, 'mxcy_base_model.pkl')
        model_path = tuned if os.path.exists(tuned) else base

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"No trained model found at: {model_path}\n"
            "Run the full pipeline (train_ml_strictly_on_real) first."
        )

    bundle = joblib.load(model_path)

    st = STOICHIOMETRIES.get(f'MC_{structure}') or \
         next((v for v in STOICHIOMETRIES.values()
               if v['structure'] == structure), None)
    if st is None:
        raise ValueError(f"Unknown structure: {structure}. "
                         f"Available: {list(CA_RATIO.keys())}")

    bl       = get_bond_length(M, C, structure)
    W        = get_bandwidth_estimate(M, C, bl, st['angle'])
    UW       = ATOMIC_DATA[M][4] / W if W != 0 else 999.0
    chi_diff = abs(ATOMIC_DATA[C][2] - ATOMIC_DATA[M][2])
    ca       = CA_RATIO[structure]
    exc_corr = get_exchange_corr_ratio(M, W)
    is_hm, spin_pol = predict_half_metal_empirical(M, C, structure, W)

    X_pred = np.array([[
        ATOMIC_DATA[M][0],
        ATOMIC_DATA[M][4],
        chi_diff,
        bl, W, UW,
        st['angle'], ca, exc_corr,
    ]])

    # ── Handle v3.9 bundle ────────────────────────────────────────────────
    if isinstance(bundle, dict) and bundle.get('version') in ('3.9', '4.0'):
        clf_metal = bundle['clf_metal']
        rf_gap    = bundle['rf_gap']
        gb_gap    = bundle['gb_gap']
        svr_gap   = bundle['svr_gap']
        rf_mag    = bundle['rf_mag']
        thresh    = bundle['insulating_threshold']

        # Stage 1: metallic?
        if clf_metal is not None:
            is_insulating = bool(clf_metal.predict(X_pred)[0])
        else:
            is_insulating = True   # assume insulating if no classifier

        # Stage 2: gap
        if is_insulating:
            gap_rf  = float(rf_gap.predict(X_pred)[0])
            gap_gb  = float(gb_gap.predict(X_pred)[0])
            gap_svr = float(svr_gap.predict(X_pred)[0]) if svr_gap else gap_rf
            # Simple average ensemble
            gap_mean = np.mean([gap_rf, gap_gb, gap_svr])
            gap_std  = np.std([gap_rf, gap_gb, gap_svr])
        else:
            gap_mean, gap_std = 0.0, 0.0

        # Magnetization (log space)
        mag_log_preds = np.array([
            e.predict(X_pred)[0] for e in rf_mag.estimators_
        ])
        mag_mean = float(np.expm1(np.mean(mag_log_preds)))
        mag_std  = float(np.expm1(np.std(mag_log_preds)))

    else:
        # ── Legacy v3.8 MultiOutputRegressor ─────────────────────────────
        model = bundle
        all_gap_preds = np.array([e.predict(X_pred)[0]
                                   for e in model.estimators_[0].estimators_])
        all_mag_preds = np.array([e.predict(X_pred)[0]
                                   for e in model.estimators_[1].estimators_])
        gap_mean = float(np.mean(all_gap_preds))
        gap_std  = float(np.std(all_gap_preds))
        mag_mean = float(np.mean(all_mag_preds))
        mag_std  = float(np.std(all_mag_preds))

    result = {
        'Compound':            f"{M}{C} ({structure})",
        'band_gap_eV':         max(0.0, gap_mean),
        'band_gap_std':        gap_std,
        'magnetization_muB':   max(0.0, mag_mean),
        'magnetization_std':   mag_std,
        'Is_Half_Metal':       is_hm,
        'Spin_Polarization_%': spin_pol,
        'U_W_ratio':           UW,
        'ca_ratio':            ca,
        'exchange_corr_ratio': exc_corr,
    }

    if verbose:
        print(f"\n{'='*55}")
        print(f"  Prediction: {result['Compound']}")
        print(f"{'='*55}")
        print(f"  Band Gap      : {result['band_gap_eV']:.3f} ± {result['band_gap_std']:.3f} eV")
        print(f"  Magnetization : {result['magnetization_muB']:.3f} ± {result['magnetization_std']:.3f} \u03bcB")
        print(f"  Half-Metal    : {'YES ✦' if is_hm else 'No'}"
              + (f"  (P = {spin_pol:.1f}%)" if is_hm else ""))
        print(f"  U/W ratio     : {UW:.2f}  ({'Mott-localised' if UW > 1 else 'itinerant'})")
        print(f"  c/a ratio     : {ca:.3f}")
        print(f"  Exch/Corr     : {exc_corr:.3f}  ({'FM instability' if exc_corr > 0.8 else 'stable'})")
        print(f"{'='*55}\n")

    return result


print("✅ predict_compound() defined (v3.9 — two-stage bundle compatible).")


---
## Cell 10 — SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) quantifies each feature's contribution  
to every individual prediction, providing model-level transparency.

> **FIX 6** — SHAP now targets the **tuned model** (same model used for predictions).  
> v2 used the base model, making importances inconsistent with actual outputs.

Output files:
- `03_SHAP_BandGap_Summary.png` — global feature importance beeswarm plot  
- `04_SHAP_cAratio_vs_BandGap.png` — dependence plot: c/a ratio effect  
- `05_SHAP_UW_vs_BandGap.png` — dependence plot: Mott criterion (U/W)


In [ ]:
def shap_analysis(model_path, df):
    """Run SHAP TreeExplainer on the band-gap sub-model and save summary plots."""
    import shap

    print("🔬 Running SHAP analysis …")
    bundle  = joblib.load(model_path)
    df_real = df[df['Is_Real']].copy()

    if len(df_real) < 5:
        print("⚠️  Not enough real data for SHAP analysis.")
        return

    X = df_real[FEATURES]
    # v4.0 bundle is a dict; extract the RF gap sub-model
    if isinstance(bundle, dict):
        gap_model = bundle['rf_gap']
    else:
        gap_model = bundle.estimators_[0]  # legacy MultiOutputRegressor
    explainer  = shap.TreeExplainer(gap_model)
    shap_vals  = explainer.shap_values(X)

    # ── Summary beeswarm plot ─────────────────────────────────────────────
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X, show=False)
    plt.title("SHAP: How Structural Features Drive the Band Gap\n"
              "(Red = high value increases gap; Blue = decreases gap)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '03_SHAP_BandGap_Summary.png'),
                dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    print("   📊 03_SHAP_BandGap_Summary.png")

    # ── c/a ratio dependence plot ─────────────────────────────────────────
    if 'ca_ratio' in FEATURES:
        plt.figure(figsize=(8, 5))
        shap.dependence_plot('ca_ratio', shap_vals, X,
                             interaction_index='U_W_ratio', show=False)
        plt.title("SHAP: Effect of c/a Ratio on Band Gap\n"
                  "(coloured by U/W ratio — verifies thesis Fig. IV.24)", fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, '04_SHAP_cAratio_vs_BandGap.png'),
                    dpi=200, bbox_inches='tight', facecolor='white')
        plt.close()
        print("   📊 04_SHAP_cAratio_vs_BandGap.png")

    # ── U/W ratio dependence plot ─────────────────────────────────────────
    plt.figure(figsize=(8, 5))
    shap.dependence_plot('U_W_ratio', shap_vals, X,
                         interaction_index='bond_angle_deg', show=False)
    plt.title("SHAP: Effect of U/W Ratio on Band Gap\n"
              "(Mott criterion: U/W > 1 → gap opens; coloured by bond angle)", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '05_SHAP_UW_vs_BandGap.png'),
                dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    print("   📊 05_SHAP_UW_vs_BandGap.png")

    print("   ✅ SHAP analysis complete.")


print("✅ shap_analysis() defined.")


---
## Cell 11 — Heatmap & Visualisation Functions

Heatmap annotation legend:
- **Bold** = real Materials Project DFT data  
- *Italic \** = empirical formula estimate  
- ✦ = predicted half-metal

> **FIX 4** applied here: `pd.isna()` replaces `np.isnan()` on the boolean  
> `Is_Half_Metal` pivot column to prevent `TypeError`.


In [ ]:
def save_fig(name):
    plt.savefig(os.path.join(OUTPUT_DIR, name), dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  📊 {name}")


def plot_heatmaps(df, prop, title, filename, cmap, vmax=None):
    """9-panel heatmap (one per stoichiometry) for a given property."""
    if vmax is None:
        vmax = df[prop].max()

    fig, axes = plt.subplots(3, 3, figsize=(22, 28))

    for ax, (stoich, _) in zip(axes.flatten(), STOICHIOMETRIES.items()):
        sub = df[df['Stoichiometry'] == stoich]

        pivot_val  = (sub.pivot_table(index='M', columns='C', values=prop)
                        .reindex(index=TM_ELEMENTS, columns=CHALCOGENS))
        pivot_real = (sub.pivot_table(index='M', columns='C',
                                      values='Is_Real', aggfunc='any')
                        .reindex(index=TM_ELEMENTS, columns=CHALCOGENS))
        pivot_hm   = (sub.pivot_table(index='M', columns='C',
                                      values='Is_Half_Metal', aggfunc='any')
                        .reindex(index=TM_ELEMENTS, columns=CHALCOGENS))

        im = ax.imshow(pivot_val.values, cmap=cmap,
                       vmin=0, vmax=vmax, aspect='auto')
        ax.set_xticks(range(len(CHALCOGENS)));   ax.set_xticklabels(CHALCOGENS, fontsize=8)
        ax.set_yticks(range(len(TM_ELEMENTS)));  ax.set_yticklabels(TM_ELEMENTS, fontsize=8)

        for i in range(len(TM_ELEMENTS)):
            for j in range(len(CHALCOGENS)):
                val = pivot_val.values[i, j]
                # FIX 4: pivot_hm.dtype may be bool — pd.isna() handles both
                #         bool and float NaN; np.isnan(bool) raises TypeError.
                is_real = not pd.isna(pivot_real.values[i, j]) and                               bool(pivot_real.values[i, j])
                is_hm_  = not pd.isna(pivot_hm.values[i, j]) and                                bool(pivot_hm.values[i, j])

                if not np.isnan(val):
                    hm_sym = "✦" if is_hm_ else ""
                    txt    = (f"{val:.1f}{hm_sym}" if is_real
                              else f"{val:.1f}*{hm_sym}")
                    weight = 'bold'   if is_real else 'normal'
                    style  = 'normal' if is_real else 'italic'
                    color  = 'white'  if val > vmax * 0.6 else 'black'
                    ax.text(j, i, txt, ha='center', va='center',
                            fontsize=8, color=color,
                            fontweight=weight, fontstyle=style)

        ax.set_title(stoich, fontweight='bold')
        plt.colorbar(im, ax=ax, shrink=0.85)

    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.03)
    fig.text(0.5, 0.99,
             "Bold = Real MP Data.   Italic * = Empirical Formula.   "
             "✦ = Predicted Half-Metal (your thesis Ch. IV)",
             ha='center', fontsize=10, style='italic', color='darkred')
    plt.tight_layout()
    save_fig(filename)


def plot_ca_ratio_effect(df):
    """Scatter: band gap and magnetization vs c/a ratio (real data only)."""
    print("  Generating c/a ratio vs band gap verification plot …")
    df_real = df[df['Is_Real']].copy()
    if len(df_real) < 3:
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc = axes[0].scatter(df_real['ca_ratio'], df_real['band_gap_eV'],
                         c=df_real['U_W_ratio'], cmap='RdYlGn',
                         s=60, alpha=0.7, edgecolors='k', linewidths=0.4)
    plt.colorbar(sc, ax=axes[0], label='U/W ratio')
    axes[0].set_xlabel('c/a ratio', fontsize=12)
    axes[0].set_ylabel('Band Gap (eV)', fontsize=12)
    axes[0].set_title('Band Gap vs c/a Ratio (Real MP Data)\n'
                      'Verifies: gap ↓ as c/a ↑ (thesis Fig. IV.24)', fontsize=10)

    sc2 = axes[1].scatter(df_real['ca_ratio'], df_real['magnetization_muB'],
                          c=df_real['d_electrons'], cmap='Blues',
                          s=60, alpha=0.7, edgecolors='k', linewidths=0.4)
    plt.colorbar(sc2, ax=axes[1], label='d electrons')
    axes[1].set_xlabel('c/a ratio', fontsize=12)
    axes[1].set_ylabel('Magnetization (μB)', fontsize=12)
    axes[1].set_title('Magnetization vs c/a Ratio (Real MP Data)\n'
                      'Verifies: moment insensitive to c/a '
                      '(thesis Fig. IV.26)', fontsize=10)

    plt.tight_layout()
    save_fig('06_cAratio_verification.png')


def plot_half_metal_map(df):
    """Heatmap of predicted spin polarisation for the ZB structure."""
    print("  Generating half-metal prediction map …")
    zb_df = df[df['Stoichiometry'] == 'MC_ZB'].copy()

    pivot = (zb_df.pivot_table(index='M', columns='C',
                                values='Spin_Polarization')
               .reindex(index=TM_ELEMENTS, columns=CHALCOGENS))

    fig, ax = plt.subplots(figsize=(9, 13))
    im = ax.imshow(pivot.values, cmap='hot_r', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(CHALCOGENS)));   ax.set_xticklabels(CHALCOGENS, fontsize=10)
    ax.set_yticks(range(len(TM_ELEMENTS)));  ax.set_yticklabels(TM_ELEMENTS, fontsize=10)

    for i in range(len(TM_ELEMENTS)):
        for j in range(len(CHALCOGENS)):
            val = pivot.values[i, j]
            if not np.isnan(val) and val > 0:
                color = 'white' if val > 60 else 'black'
                ax.text(j, i, f"{val:.0f}%", ha='center', va='center',
                        fontsize=10, color=color, fontweight='bold')

    plt.colorbar(im, ax=ax, label='Predicted Spin Polarization (%)')
    ax.set_title("Half-Metal Prediction Map — ZB Structure\n"
                 "Confirms thesis (Ch. IV.3.6, IV.4.5): "
                 "CrS, FeS, CoS, NiS are half-metallic in ZB",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    save_fig('07_halfmetal_map_ZB.png')


print("✅ Plotting functions defined.")


---
## Cell 12 — Excel Export

Saves five worksheets to `MxCy_v3_Full_Results.xlsx`:
1. **Dataset v3** — full feature + target table  
2. **Feature Importance (Real)** — SHAP-consistent importances  
3. **ML CV Results** — cross-validation R² scores  
4. **Half-Metal Predictions** — predicted half-metals with spin polarisation  
5. **Feature Rationale** — physics basis and thesis/article references


In [ ]:
def export_excel(df, imp, cv_results):
    """Export all results to a multi-sheet Excel workbook."""
    wb = Workbook()

    ws = wb.active
    ws.title = "Dataset v3"
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)

    if not imp.empty:
        ws2 = wb.create_sheet("Feature Importance (Real)")
        for row in dataframe_to_rows(imp, index=False, header=True):
            ws2.append(row)

    if cv_results:
        ws3 = wb.create_sheet("ML CV Results")
        ws3.append(["Metric", "Value"])
        for k, v in cv_results.items():
            ws3.append([k, round(v, 4) if isinstance(v, float) else v])

    ws4 = wb.create_sheet("Half-Metal Predictions")
    hm_df = df[df['Is_Half_Metal']].copy()
    for row in dataframe_to_rows(
            hm_df[['Compound', 'M', 'C', 'Stoichiometry',
                   'Spin_Polarization', 'U_W_ratio',
                   'exchange_corr_ratio']].reset_index(drop=True),
            index=False, header=True):
        ws4.append(row)

    ws5 = wb.create_sheet("Feature Rationale")
    ws5.append(["Feature", "Physics Basis", "Source"])
    for row in [
        ["d_electrons",         "Crystal field splitting, Hund's rule",      "Thesis Ch.I, Tab.I.6"],
        ["Hubbard_U",           "On-site Coulomb repulsion — Mott criterion", "Article Eq.1-2"],
        ["chi_diff",            "Ionicity → charge-transfer gap",             "Thesis Tab.II.3"],
        ["bond_length_A",       "Orbital overlap → bandwidth",                "Thesis Tab.II.4"],
        ["bandwidth_W",         "d-band width → itinerancy vs localisation",  "Article Sec.4.2"],
        ["U_W_ratio",           "Mott criterion: U/W>1 → insulating",         "Thesis Ch.II.3.2"],
        ["bond_angle_deg",      "Goodenough–Kanamori → FM vs AFM",            "Thesis Ch.II.4"],
        ["ca_ratio",            "Interlayer distance → gap (inverse)",        "Article Fig.9, Thesis Fig.IV.24"],
        ["exchange_corr_ratio", "Stoner I/W → exchange vs correlation",       "Article Sec.4.2 (α role)"],
    ]:
        ws5.append(row)

    path = os.path.join(OUTPUT_DIR, 'MxCy_v3_Full_Results.xlsx')
    wb.save(path)
    print(f"✅ Excel saved: {path}")


print("✅ export_excel() defined.")


---
## Cell 13 — Run Full Pipeline

Execute the complete workflow end-to-end.  
Individual steps can be re-run independently after the first execution.

**v4.0:** `compare_predictions_vs_real()` is called after training (Step 3b) to generate all scatter / Bland-Altman / confusion-matrix / per-compound plots and a `Prediction_vs_Real` Excel sheet.


In [ ]:
print("=" * 65)
print("  MxCy Pipeline v4.0 — Physics-Constrained ML (Expanded Data)")
print("  Based on: Reggad et al., Physica B 526 (2017) 89-95")
print("  and: Reggad, These de Doctorat, Univ. Sidi-Bel-Abbes")
print("=" * 65)

try:
    # Step 1 — Fetch real DFT data (v4.0: hull≤0.15, ordering field)
    real_dict = fetch_real_data()

    # Step 2 — Build full dataset (real + empirical)
    df_full = build_full_dataset(real_dict)

    # Propagate ordering columns to full dataset for real rows
    if real_dict:
        ord_label_map = {(M, C, sk): v['ordering_label']
                         for (M, C, sk), v in real_dict.items()}
        ord_str_map   = {(M, C, sk): v['ordering_str']
                         for (M, C, sk), v in real_dict.items()}
        df_full['ordering_label'] = df_full.apply(
            lambda r: ord_label_map.get((r['M'], r['C'], r['Stoichiometry']), 0)
            if r['Is_Real'] else -1, axis=1)
        df_full['ordering_str'] = df_full.apply(
            lambda r: ord_str_map.get((r['M'], r['C'], r['Stoichiometry']), 'NM')
            if r['Is_Real'] else 'Unknown', axis=1)

    # Step 3 — Train ML strictly on real data (v4.0: returns df_test)
    importance_df, cv_results, model_path, df_test = train_ml_strictly_on_real(df_full)

    # Step 3b — Compare predictions vs real on held-out test set
    comparison_df = pd.DataFrame()
    if model_path and len(df_test) >= 10:
        comparison_df = compare_predictions_vs_real(df_test, cv_results)

    # Step 4 — Fine-tune by blending empirical physics trees
    if model_path:
        tune_with_empirical_physics(model_path, df_full)

    # Step 5 — Example predictions (thesis compounds)
    print("\n=== Example Predictions (Thesis Compounds) ===")
    for M, C, struct in [
        ('Ni', 'S',  'NiAs'),
        ('Ni', 'S',  'zincblende'),
        ('Cr', 'S',  'zincblende'),
        ('Fe', 'S',  'zincblende'),
        ('Co', 'S',  'zincblende'),
        ('Ni', 'S',  'MnP'),
    ]:
        predict_compound(M, C, struct)

    # Step 6 — SHAP analysis
    shap_model_path = os.path.join(OUTPUT_DIR, 'mxcy_tuned_model.pkl')
    if not os.path.exists(shap_model_path):
        shap_model_path = model_path
    if shap_model_path:
        shap_analysis(shap_model_path, df_full)

    # Step 7 — Heatmaps
    print("\nGenerating heatmaps …")
    plot_heatmaps(df_full, 'band_gap_eV',      'Band Gap (eV)',      '01_band_gap_v4.png',      'RdYlGn_r')
    plot_heatmaps(df_full, 'magnetization_muB', 'Magnetization (μB)', '02_magnetization_v4.png', 'Blues')
    plot_ca_ratio_effect(df_full)
    plot_half_metal_map(df_full)

    # Step 8 — Export Excel (v4.0: add Prediction_vs_Real sheet)
    export_excel(df_full, importance_df, cv_results)

    if not comparison_df.empty:
        from openpyxl import load_workbook
        from openpyxl.utils.dataframe import dataframe_to_rows
        xl_path = os.path.join(OUTPUT_DIR, 'MxCy_v3_Full_Results.xlsx')
        wb_app = load_workbook(xl_path)
        ws_cmp = wb_app.create_sheet("Prediction_vs_Real")
        ws_cmp.append(["Prediction vs Real — Held-Out Test Set"])
        ws_cmp.append([])
        for row in dataframe_to_rows(comparison_df, index=False, header=True):
            ws_cmp.append(row)
        wb_app.save(xl_path)
        print(f"✅ Prediction_vs_Real sheet added to {xl_path}")

    print("=" * 65)
    print(f"✅ All outputs saved to: {OUTPUT_DIR}")
    print("   Key output files:")
    print("   08_PredVsReal_BandGap.png        — scatter + Bland-Altman (gap)")
    print("   09_PredVsReal_Magnetization.png  — scatter + Bland-Altman (mag)")
    print("   10_Ordering_ConfusionMatrix.png  — NM/AFM/FM prediction accuracy")
    print("   11_PerCompound_Comparison.png    — per-compound gap bar chart")
    print("   12_PerCompound_Magnetization.png — per-compound mag bar chart")
    print("   MxCy_v3_Full_Results.xlsx        — incl. Prediction_vs_Real sheet")
    print("=" * 65)

    # Step 9 — Auto-download (Colab only)
    if PLATFORM == "colab":
        print("\n📥 Downloading output files to your computer …")
        import shutil
        from google.colab import files as _colab_files
        zip_path = "/content/MxCy_results.zip"
        shutil.make_archive("/content/MxCy_results", "zip", OUTPUT_DIR)
        _colab_files.download(zip_path)
        print("   ✅ MxCy_results.zip downloaded.")

except Exception as e:
    import traceback
    print(f"\n❌ Error: {e}")
    traceback.print_exc()
